# New Cluster Analysis
Reads final hotel clusters from `s3a://delta-lake-mumbai/bronze/final_clusters/` and explores their structure and statistics.

In [1]:

# cell 1
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, count, avg, round as spark_round, min as spark_min, max as spark_max
import os

# Override any problematic environment variables
os.environ.pop('HADOOP_CONF_DIR', None)
os.environ.pop('YARN_CONF_DIR', None)

# Create Spark session with MinIO/S3A + Delta Lake configuration
# Spark 4.0.1 uses Scala 2.13 → requires Delta Lake 4.x (Scala 2.13 build)
spark = SparkSession.builder \
    .appName("Analyze Hotel Pairs") \
    .config("spark.jars.packages",
            "org.apache.hadoop:hadoop-aws:3.3.4,"
            "com.amazonaws:aws-java-sdk-bundle:1.12.262,"
            "io.delta:delta-spark_2.13:4.0.0") \
    .config("spark.sql.extensions", "io.delta.sql.DeltaSparkSessionExtension") \
    .config("spark.sql.catalog.spark_catalog", "org.apache.spark.sql.delta.catalog.DeltaCatalog") \
    .config("spark.hadoop.fs.s3a.endpoint", "http://localhost:9000") \
    .config("spark.hadoop.fs.s3a.access.key", "minioadmin") \
    .config("spark.hadoop.fs.s3a.secret.key", "minioadmin") \
    .config("spark.hadoop.fs.s3a.path.style.access", "true") \
    .config("spark.hadoop.fs.s3a.impl", "org.apache.hadoop.fs.s3a.S3AFileSystem") \
    .config("spark.hadoop.fs.s3a.connection.ssl.enabled", "false") \
    .config("spark.hadoop.fs.s3a.aws.credentials.provider", "org.apache.hadoop.fs.s3a.SimpleAWSCredentialsProvider") \
    .config("spark.hadoop.fs.s3a.connection.timeout", "60000") \
    .config("spark.hadoop.fs.s3a.connection.establish.timeout", "60000") \
    .config("spark.hadoop.fs.s3a.socket.send.buffer", "8192") \
    .config("spark.hadoop.fs.s3a.socket.recv.buffer", "8192") \
    .config("spark.hadoop.fs.s3a.attempts.maximum", "3") \
    .config("spark.hadoop.fs.s3a.retry.limit", "3") \
    .config("spark.hadoop.fs.s3a.retry.interval", "500") \
    .config("spark.hadoop.fs.s3a.retry.throttle.limit", "3") \
    .config("spark.hadoop.fs.s3a.retry.throttle.interval", "1000") \
    .config("spark.hadoop.fs.s3a.connection.maximum", "50") \
    .config("spark.hadoop.fs.s3a.threads.max", "10") \
    .config("spark.hadoop.fs.s3a.threads.core", "5") \
    .config("spark.hadoop.fs.s3a.threads.keepalivetime", "60") \
    .config("spark.hadoop.fs.s3a.max.total.tasks", "5") \
    .config("spark.hadoop.fs.s3a.readahead.range", "65536") \
    .config("spark.hadoop.fs.s3a.paging.maximum", "5") \
    .config("spark.hadoop.fs.s3a.list.version", "2") \
    .config("spark.hadoop.fs.s3a.committer.threads", "4") \
    .config("spark.hadoop.fs.s3a.committer.staging.tmp.path", "/tmp/staging") \
    .config("spark.hadoop.fs.s3a.buffer.dir", "/tmp") \
    .config("spark.hadoop.fs.s3a.multipart.size", "104857600") \
    .config("spark.hadoop.fs.s3a.multipart.threshold", "2147483647") \
    .config("spark.hadoop.fs.s3a.multipart.purge.age", "86400") \
    .config("spark.hadoop.fs.s3a.fast.upload", "true") \
    .config("spark.hadoop.fs.s3a.fast.upload.buffer", "disk") \
    .config("spark.hadoop.fs.s3a.fast.upload.active.blocks", "4") \
    .config("spark.hadoop.fs.s3a.block.size", "33554432") \
    .config("spark.hadoop.fs.s3a.metadatastore.authoritative", "false") \
    .config("spark.sql.files.maxPartitionBytes", "134217728") \
    .config("spark.driver.memory", "2g") \
    .config("spark.ui.port", "4050") \
    .getOrCreate()

spark.sparkContext.setLogLevel("WARN")
print("✓ Spark UI available at: http://localhost:4050")
print("✓ Spark session created successfully")


:: loading settings :: url = jar:file:/Library/Frameworks/Python.framework/Versions/3.13/lib/python3.13/site-packages/pyspark/jars/ivy-2.5.3.jar!/org/apache/ivy/core/settings/ivysettings.xml
Ivy Default Cache set to: /Users/nakul.patil/.ivy2.5.2/cache
The jars for the packages stored in: /Users/nakul.patil/.ivy2.5.2/jars
org.apache.hadoop#hadoop-aws added as a dependency
com.amazonaws#aws-java-sdk-bundle added as a dependency
io.delta#delta-spark_2.13 added as a dependency
:: resolving dependencies :: org.apache.spark#spark-submit-parent-04d2ad5c-b74e-44c2-8851-b519ef74fb7b;1.0
	confs: [default]
	found org.apache.hadoop#hadoop-aws;3.3.4 in central
	found com.amazonaws#aws-java-sdk-bundle;1.12.262 in central
	found org.wildfly.openssl#wildfly-openssl;1.0.7.Final in local-m2-cache
	found io.delta#delta-spark_2.13;4.0.0 in central
	found io.delta#delta-storage;4.0.0 in central
	found org.antlr#antlr4-runtime;4.13.1 in central
:: resolution report :: resolve 161ms :: artifacts dl 5ms
	:: m

✓ Spark UI available at: http://localhost:4050
✓ Spark session created successfully


## 2. Load Final Clusters

In [2]:

# cell 2
from pyspark.sql.functions import col, count

CLUSTERS_PATH = "s3a://delta-lake/bronze/final_clusters/"

# this is new cluster data after hotel mapping
cluster_data = spark.read.format("delta").load(CLUSTERS_PATH)


print(f"✓ Loaded clusters from: {CLUSTERS_PATH}")
print(f"  Total rows : {cluster_data.count():,}")
print(f"  Columns    : {len(cluster_data.columns)}")
print()
cluster_data.printSchema()

# ── Duplicate check on uid ────────────────────────────────────────────────────
_total    = cluster_data.count()
_distinct = cluster_data.select("uid").distinct().count()
_dup      = _total - _distinct
print(f"Duplicate check — total: {_total:,}  |  distinct uid: {_distinct:,}  |  duplicate uid rows: {_dup:,}")
if _dup > 0:
    print("  Top duplicate uids:")
    cluster_data \
        .groupBy("uid") \
        .agg(count("*").alias("cnt")) \
        .filter(col("cnt") > 1) \
        .orderBy(col("cnt").desc()) \
        .show(10, truncate=False)

cluster_data = cluster_data.dropDuplicates(["uid"])
print(f"After dedup — {cluster_data.count():,} distinct cluster rows carried forward.")


26/03/14 21:21:23 WARN MetricsConfig: Cannot locate configuration: tried hadoop-metrics2-s3a-file-system.properties,hadoop-metrics2.properties


✓ Loaded clusters from: s3a://delta-lake/bronze/final_clusters/


26/03/14 21:21:26 WARN SparkStringUtils: Truncated the string representation of a plan since it was too large. This behavior can be adjusted by setting 'spark.sql.debug.maxToStringFields'.


  Total rows : 162,708
  Columns    : 4

root
 |-- cluster_id: string (nullable = true)
 |-- name: string (nullable = true)
 |-- uid: string (nullable = true)
 |-- providerName: string (nullable = true)



Duplicate check — total: 162,708  |  distinct uid: 162,708  |  duplicate uid rows: 0
After dedup — 162,708 distinct cluster rows carried forward.


In [3]:

# cell 3
# this is hotel input data after normalization and mapping
from pyspark.sql.functions import col, count

mapped_path = "s3a://delta-lake/bronze/hotels"
print(f"Reading from: {mapped_path}")

hotel_data = spark.read.format("delta").load(mapped_path)

print(f"\n✓ Data loaded successfully")
print(f"Total records : {hotel_data.count():,}")
print(f"Total columns : {len(hotel_data.columns)}")

hotel_data.printSchema()
hotel_data.show(20, truncate=False)

# ── Duplicate check on uid ────────────────────────────────────────────────────
_total    = hotel_data.count()
_distinct = hotel_data.select("uid").distinct().count()
_dup      = _total - _distinct
print(f"\nDuplicate check — total: {_total:,}  |  distinct uid: {_distinct:,}  |  duplicate uid rows: {_dup:,}")
if _dup > 0:
    print("  Top duplicate uids:")
    hotel_data \
        .groupBy("uid") \
        .agg(count("*").alias("cnt")) \
        .filter(col("cnt") > 1) \
        .orderBy(col("cnt").desc()) \
        .show(10, truncate=False)

hotel_data = hotel_data.dropDuplicates(["uid"])
print(f"After dedup — {hotel_data.count():,} distinct hotels carried forward.")


Reading from: s3a://delta-lake/bronze/hotels

✓ Data loaded successfully
Total records : 162,299
Total columns : 35
root
 |-- id: string (nullable = true)
 |-- name: string (nullable = true)
 |-- relevanceScore: string (nullable = true)
 |-- providerId: string (nullable = true)
 |-- providerHotelId: string (nullable = true)
 |-- providerName: string (nullable = true)
 |-- language: string (nullable = true)
 |-- geoCode: struct (nullable = true)
 |    |-- lat: string (nullable = true)
 |    |-- long: string (nullable = true)
 |-- contact: struct (nullable = true)
 |    |-- address: struct (nullable = true)
 |    |    |-- line1: string (nullable = true)
 |    |    |-- city: struct (nullable = true)
 |    |    |    |-- name: string (nullable = true)
 |    |    |-- state: struct (nullable = true)
 |    |    |    |-- name: string (nullable = true)
 |    |    |-- country: struct (nullable = true)
 |    |    |    |-- code: string (nullable = true)
 |    |    |    |-- name: string (nullable = 

+--------+--------------------------------------------------+--------------+----------+---------------+------------+--------+------------------------+-----------------------------------------------------------------------------------------------------------------------------------------------+---------------------+---------------------+----------+--------+----------+----------+------------------+-----------+------------+-------------------------------------+-------------------------+--------------------------+----------------------------+----------------------------+--------------------------+----------------+----------------+-------------------+--------------------------+-------------------------------------------------------------------------------------------+----------------------------------------------------------------+--------------------------------------------+---------------------------------------------------------------------------------------------------------------------


Duplicate check — total: 162,299  |  distinct uid: 161,871  |  duplicate uid rows: 428
  Top duplicate uids:


+----------------------------------------------------------------+---+
|uid                                                             |cnt|
+----------------------------------------------------------------+---+
|68195242ff19b917aab2e28274456e8906ebc7e51d3106a6eac37c843d2e886f|17 |
|05af9ffcd57224a358d590cdc6365946c22ac4ca5b876cfca772649007a173fe|15 |
|f9794b6a7a5d591df155854a477ea8995c5d19c5ad0c562027a9f17aeac5e3a0|13 |
|5d77615f54fc0b0b0c17dd83c590df52baf1d7c388e0953823d02f1ac076b789|11 |
|495ea301e2b2f07df49f598ab9a2531f6629cf7cfc0562bb335b834485a36fd8|8  |
|dbde436f5dbebb0d1d99f530777dfd771eb9a5bdb1df6a003b058fc96b2e88ee|7  |
|4e588f8c2af7b3a6522b72419eb612cad040b74c7ebdd205cb2fde244633e16f|6  |
|7a3b8f6d772b588a3927b0e0cdfadefc34675b01a7b6e68fc234f9c530cbc158|6  |
|bb443509b0b274eff7a3302ee1aa2655874d4bcf2e7b2cc152bf22b9c8d0791d|5  |
|981aa3797736303fbe2c79a9be204d86eaf033223bc353f4fd3d8df21b754c31|5  |
+----------------------------------------------------------------+---+
only s

After dedup — 161,871 distinct hotels carried forward.


In [4]:

# cell 4
# this is hotel pair data which has all scores information for pairs
from pyspark.sql.functions import least, greatest, when, col, count

paired_path = "s3a://delta-lake/bronze/hotel_pairs"
print(f"Reading from: {paired_path}")

hotel_pairs = spark.read.format("delta").load(paired_path)

print(f"\n✓ Data loaded successfully")
print(f"Total records : {hotel_pairs.count():,}")
print(f"Total columns : {len(hotel_pairs.columns)}")

hotel_pairs.printSchema()
hotel_pairs.show(20, truncate=False)

# ── Duplicate check on raw parquet ───────────────────────────────────────────
# Canonical key: least/greatest so (X,Y) and (Y,X) count as the same pair
_total_raw    = hotel_pairs.count()
_distinct_raw = hotel_pairs \
    .select(least("uid_i","uid_j").alias("k1"), greatest("uid_i","uid_j").alias("k2")) \
    .distinct().count()
_dup_raw = _total_raw - _distinct_raw
print(f"\nRaw parquet  — total: {_total_raw:,}  |  distinct pairs (canonical): {_distinct_raw:,}  |  duplicates (incl. mirrors): {_dup_raw:,}")
if _dup_raw > 0:
    print("  Top duplicate canonical pairs (X<>Y same as Y<>X):")
    hotel_pairs \
        .withColumn("k1", least("uid_i","uid_j")) \
        .withColumn("k2", greatest("uid_i","uid_j")) \
        .groupBy("k1", "k2") \
        .agg(count("*").alias("cnt")) \
        .filter(col("cnt") > 1) \
        .orderBy(col("cnt").desc()) \
        .show(10, truncate=False)

# ── Normalise to canonical direction, then deduplicate ────────────────────────
# Problem with plain filter(uid_i < uid_j): pairs that exist ONLY in the (Y,X)
# direction get silently dropped, so canonical count != deduped count.
# Fix: swap all _i / _j columns when uid_i > uid_j, then dropDuplicates.

cols_i = {c[:-2] for c in hotel_pairs.columns if c.endswith("_i")}
cols_j = {c[:-2] for c in hotel_pairs.columns if c.endswith("_j")}
paired_bases = cols_i & cols_j

needs_swap = col("uid_i") > col("uid_j")

select_exprs = []
for c in hotel_pairs.columns:
    if c.endswith("_i") and c[:-2] in paired_bases:
        base = c[:-2]
        select_exprs.append(
            when(needs_swap, col(f"{base}_j")).otherwise(col(f"{base}_i")).alias(f"{base}_i")
        )
    elif c.endswith("_j") and c[:-2] in paired_bases:
        base = c[:-2]
        select_exprs.append(
            when(needs_swap, col(f"{base}_i")).otherwise(col(f"{base}_j")).alias(f"{base}_j")
        )
    else:
        select_exprs.append(col(c))

hotel_pairs = hotel_pairs.select(select_exprs).dropDuplicates(["uid_i", "uid_j"])
print(f"After normalise+dedup — {hotel_pairs.count():,} distinct canonical pairs carried forward.")


Reading from: s3a://delta-lake/bronze/hotel_pairs

✓ Data loaded successfully
Total records : 326,192
Total columns : 42
root
 |-- id_i: string (nullable = true)
 |-- id_j: string (nullable = true)
 |-- uid_i: string (nullable = true)
 |-- uid_j: string (nullable = true)
 |-- providerName_i: string (nullable = true)
 |-- providerName_j: string (nullable = true)
 |-- providerHotelId_i: string (nullable = true)
 |-- providerHotelId_j: string (nullable = true)
 |-- name_i: string (nullable = true)
 |-- name_j: string (nullable = true)
 |-- normalized_name_i: string (nullable = true)
 |-- normalized_name_j: string (nullable = true)
 |-- type_i: string (nullable = true)
 |-- type_j: string (nullable = true)
 |-- geo_distance_km: double (nullable = true)
 |-- name_score_containment: float (nullable = true)
 |-- normalized_name_score_containment: float (nullable = true)
 |-- name_score_jaccard: float (nullable = true)
 |-- normalized_name_score_jaccard: float (nullable = true)
 |-- name_score


Raw parquet  — total: 326,192  |  distinct pairs (canonical): 233,476  |  duplicates (incl. mirrors): 92,716
  Top duplicate canonical pairs (X<>Y same as Y<>X):


+----------------------------------------------------------------+----------------------------------------------------------------+-----+
|k1                                                              |k2                                                              |cnt  |
+----------------------------------------------------------------+----------------------------------------------------------------+-----+
|68195242ff19b917aab2e28274456e8906ebc7e51d3106a6eac37c843d2e886f|f9794b6a7a5d591df155854a477ea8995c5d19c5ad0c562027a9f17aeac5e3a0|48841|
|05af9ffcd57224a358d590cdc6365946c22ac4ca5b876cfca772649007a173fe|4e588f8c2af7b3a6522b72419eb612cad040b74c7ebdd205cb2fde244633e16f|8100 |
|68195242ff19b917aab2e28274456e8906ebc7e51d3106a6eac37c843d2e886f|bb443509b0b274eff7a3302ee1aa2655874d4bcf2e7b2cc152bf22b9c8d0791d|7225 |
|bb443509b0b274eff7a3302ee1aa2655874d4bcf2e7b2cc152bf22b9c8d0791d|f9794b6a7a5d591df155854a477ea8995c5d19c5ad0c562027a9f17aeac5e3a0|4225 |
|05af9ffcd57224a358d590cdc6365946c

In [5]:

# cell 5 — enrich hotel_pairs with hotel_data and cluster_data columns
# (hotel_pairs is already normalised + deduped in cell 4)

# Columns to bring in from hotel_data / cluster_data for each entity
HOTEL_JOIN_COLS = [
    "geoCode_lat",
    "geoCode_long",
    "combined_address",
    "contact_phones",
    "contact_emails",
    "starRating",
]
CLUSTER_JOIN_COLS = ["cluster_id"]

original_cols = list(hotel_pairs.columns)
existing_cols = set(original_cols)

# Register temp views — SQL table aliases make column refs unambiguous
hotel_pairs.createOrReplaceTempView("hp_base")
hotel_data.createOrReplaceTempView("hotel_data_v")
cluster_data.createOrReplaceTempView("cluster_data_v")

# Only include columns not already present (belt-and-suspenders guard)
extra_parts = (
    [f"h_i.{c} AS {c}_i"  for c in HOTEL_JOIN_COLS   if f"{c}_i"  not in existing_cols] +
    [f"h_j.{c} AS {c}_j"  for c in HOTEL_JOIN_COLS   if f"{c}_j"  not in existing_cols] +
    [f"c_i.{c} AS {c}_i"  for c in CLUSTER_JOIN_COLS  if f"{c}_i"  not in existing_cols] +
    [f"c_j.{c} AS {c}_j"  for c in CLUSTER_JOIN_COLS  if f"{c}_j"  not in existing_cols]
)
extra_sql = (", " + ", ".join(extra_parts)) if extra_parts else ""

enriched = spark.sql(f"""
    SELECT hp.*{extra_sql}
    FROM hp_base hp
    LEFT JOIN hotel_data_v   h_i ON hp.uid_i = h_i.uid
    LEFT JOIN hotel_data_v   h_j ON hp.uid_j = h_j.uid
    LEFT JOIN cluster_data_v c_i ON hp.uid_i = c_i.uid
    LEFT JOIN cluster_data_v c_j ON hp.uid_j = c_j.uid
""")

# Reorder extra columns as: X_i, X_j, Y_i, Y_j, ...
extra_ordered = []
for base in [*HOTEL_JOIN_COLS, *CLUSTER_JOIN_COLS]:
    for suffix in ("i", "j"):
        c = f"{base}_{suffix}"
        if c in enriched.columns:
            extra_ordered.append(c)

orig_kept   = [c for c in original_cols if c not in extra_ordered]
final_order = [c for c in orig_kept + extra_ordered if c in enriched.columns]
hotel_pairs = enriched.select(*final_order)

print(f"✓ hotel_pairs enriched: {hotel_pairs.count():,} rows, {len(hotel_pairs.columns)} columns")
print()
print("All columns after joins:")
for c in hotel_pairs.columns:
    print(f"  {c}")

hotel_pairs.select(*extra_ordered).show(10, truncate=50)


✓ hotel_pairs enriched: 233,476 rows, 56 columns

All columns after joins:
  id_i
  id_j
  uid_i
  uid_j
  providerName_i
  providerName_j
  providerHotelId_i
  providerHotelId_j
  name_i
  name_j
  normalized_name_i
  normalized_name_j
  type_i
  type_j
  geo_distance_km
  name_score_containment
  normalized_name_score_containment
  name_score_jaccard
  normalized_name_score_jaccard
  name_score_lcs
  normalized_name_score_lcs
  name_score_levenshtein
  normalized_name_score_levenshtein
  name_score_sbert
  normalized_name_score_sbert
  average_name_score
  average_normalized_name_score
  address_line1_score
  postal_code_match
  country_match
  address_sbert_score
  phone_match_score
  email_match_score
  fax_match_score
  property_type_score
  name_unit_score
  address_unit_score
  supplier_score
  star_ratings_score
  overall_pair_score
  pair_key_left
  pair_key_right
  geoCode_lat_i
  geoCode_lat_j
  geoCode_long_i
  geoCode_long_j
  combined_address_i
  combined_address_j
  cont

+-------------+-------------+--------------+--------------+--------------------------------------------------+--------------------------------------------------+----------------+----------------+----------------+----------------+------------+------------+------------+------------+
|geoCode_lat_i|geoCode_lat_j|geoCode_long_i|geoCode_long_j|                                combined_address_i|                                combined_address_j|contact_phones_i|contact_phones_j|contact_emails_i|contact_emails_j|starRating_i|starRating_j|cluster_id_i|cluster_id_j|
+-------------+-------------+--------------+--------------+--------------------------------------------------+--------------------------------------------------+----------------+----------------+----------------+----------------+------------+------------+------------+------------+
|     33.82568|     33.82605|     -78.65073|     -78.64949|1806 north ocean blvd north myrtle beach south ...|1820 north ocean blvd north myrtle beach sou

In [6]:
# cell 6
import yaml
from pyspark.sql.functions import col, when, sum as spark_sum, expr

CONFIG_PATH = "/Users/nakul.patil/Documents/hotel-mapping/src/hotel_data/config/config.yaml"

# ── helpers ───────────────────────────────────────────────────────────────────

CMP_MAP = {"gte": ">=", "lte": "<=", "gt": ">", "lt": "<", "eq": "=", "neq": "!="}

def build_readable(node, indent=0):
    """Recursively build an indented human-readable expression."""
    pad = "  " * indent
    if "signal" in node:
        cmp = CMP_MAP.get(node["comparator"], node["comparator"])
        return f"{pad}{node['signal']} {cmp} {node['threshold']}"
    op    = node["operator"].upper()
    parts = [build_readable(r, indent + 1) for r in node["rules"]]
    joiner = f"\n{pad}{op}\n"
    inner  = joiner.join(parts)
    return f"{pad}(\n{inner}\n{pad})"

def build_spark_expr(node):
    """Build a flat Spark SQL expression string."""
    if "signal" in node:
        cmp = CMP_MAP.get(node["comparator"], node["comparator"])
        return f"({node['signal']} {cmp} {node['threshold']})"
    op    = f" {node['operator'].upper()} "
    parts = [build_spark_expr(r) for r in node["rules"]]
    return "(" + op.join(parts) + ")"

# ── 1. load config & build expressions ───────────────────────────────────────

with open(CONFIG_PATH) as f:
    cfg = yaml.safe_load(f)

match_logic    = cfg["scoring"]["match_logic"]
human_readable = build_readable(match_logic)
spark_filter   = build_spark_expr(match_logic)

# ── 2. print the WHERE clause ─────────────────────────────────────────────────

print("=" * 70)
print("WHERE CLAUSE (human-readable)")
print("=" * 70)
print(human_readable)
print()

# ── 3. apply WHERE clause and check alignment with cluster_id_i = cluster_id_j ─

df = hotel_pairs \
    .withColumn("rule_match",   expr(spark_filter)) \
    .withColumn("same_cluster", col("cluster_id_i") == col("cluster_id_j"))

counts = df.agg(
    spark_sum(when( col("rule_match") &  col("same_cluster"), 1).otherwise(0)).alias("TP"),
    spark_sum(when( col("rule_match") & ~col("same_cluster"), 1).otherwise(0)).alias("FP"),
    spark_sum(when(~col("rule_match") &  col("same_cluster"), 1).otherwise(0)).alias("FN"),
    spark_sum(when(~col("rule_match") & ~col("same_cluster"), 1).otherwise(0)).alias("TN"),
).collect()[0]

aln_TP, aln_FP, aln_FN, aln_TN = counts["TP"], counts["FP"], counts["FN"], counts["TN"]
aln_total = aln_TP + aln_FP + aln_FN + aln_TN

aln_precision = aln_TP / (aln_TP + aln_FP) if (aln_TP + aln_FP) > 0 else 0.0
aln_recall    = aln_TP / (aln_TP + aln_FN) if (aln_TP + aln_FN) > 0 else 0.0
aln_f1        = 2 * aln_precision * aln_recall / (aln_precision + aln_recall) if (aln_precision + aln_recall) > 0 else 0.0

print("=" * 70)
print("ALIGNMENT CHECK  (rule_match  vs  cluster_id_i == cluster_id_j)")
print("=" * 70)
print(f"  Total pairs                : {aln_total:>10,}")
print(f"  TP (rule=✓, cluster=same)  : {aln_TP:>10,}  ← correctly matched")
print(f"  FP (rule=✓, cluster=diff)  : {aln_FP:>10,}  ← over-matched by rule")
print(f"  FN (rule=✗, cluster=same)  : {aln_FN:>10,}  ← missed by rule")
print(f"  TN (rule=✗, cluster=diff)  : {aln_TN:>10,}  ← correctly excluded")
print()
print(f"  Precision : {aln_precision:.4f}")
print(f"  Recall    : {aln_recall:.4f}")
print(f"  F1        : {aln_f1:.4f}")


WHERE CLAUSE (human-readable)
(
  (
    name_score_jaccard >= 0.9
  OR
    name_score_lcs >= 0.9
  OR
    name_score_sbert >= 0.9
  OR
    name_score_levenshtein >= 0.95
  OR
    normalized_name_score_jaccard >= 0.95
  OR
    normalized_name_score_lcs >= 0.95
  OR
    normalized_name_score_levenshtein >= 0.95
  OR
    normalized_name_score_sbert >= 0.95
  OR
    average_name_score >= 0.8
  OR
    average_normalized_name_score >= 0.9
  OR
    (
      normalized_name_score_containment >= 1
    AND
      name_score_sbert >= 0.75
    AND
      geo_distance_km <= 0.2
    )
  OR
    (
      normalized_name_score_containment >= 1
    AND
      name_score_sbert >= 0.7
    AND
      geo_distance_km <= 0.1
    )
  OR
    (
      name_score_containment >= 0.95
    AND
      normalized_name_score_containment >= 0.95
    AND
      star_ratings_score >= 0.8
    AND
      geo_distance_km <= 0.2
    )
  )
AND
  (
    address_line1_score >= 0.5
  OR
    address_sbert_score >= 0.5
  OR
    geo_distance_

ALIGNMENT CHECK  (rule_match  vs  cluster_id_i == cluster_id_j)
  Total pairs                :    233,476
  TP (rule=✓, cluster=same)  :     37,457  ← correctly matched
  FP (rule=✓, cluster=diff)  :        129  ← over-matched by rule
  FN (rule=✗, cluster=same)  :        161  ← missed by rule
  TN (rule=✗, cluster=diff)  :    195,729  ← correctly excluded

  Precision : 0.9966
  Recall    : 0.9957
  F1        : 0.9961


In [7]:
# cell 7
# ── Old vs New cluster logic assessment ──────────────────────────────────────
# Old logic : id_i == id_j         → these two hotels were matched before
# New logic : cluster_id_i == cluster_id_j → these two hotels are matched now
#
#  TP  old=match  new=match   → both agree it's a match
#  FN  old=match  new=miss    → new clustering LOST a match the old had
#  FP  old=miss   new=match   → new clustering GAINED a match the old didn't have
#  TN  old=miss   new=miss    → both agree it's not a match

import os, datetime
from pyspark.sql.functions import col, when, sum as spark_sum

REPORTS_DIR = os.path.expanduser("~/Downloads/hotel_mapping_reports")
os.makedirs(REPORTS_DIR, exist_ok=True)
if "RUN_TIMESTAMP" not in globals():
    RUN_TIMESTAMP = datetime.datetime.now().strftime("%Y%m%d_%H%M%S")

FP_CSV = os.path.join(REPORTS_DIR, f"cluster_fp_{RUN_TIMESTAMP}.csv")
FN_CSV = os.path.join(REPORTS_DIR, f"cluster_fn_{RUN_TIMESTAMP}.csv")

df = hotel_pairs \
    .withColumn("old_match", col("id_i") == col("id_j")) \
    .withColumn("new_match", col("cluster_id_i") == col("cluster_id_j"))

counts = df.agg(
    spark_sum(when(col("old_match") &  col("new_match"), 1).otherwise(0)).alias("TP"),
    spark_sum(when(col("old_match") & ~col("new_match"), 1).otherwise(0)).alias("FN"),
    spark_sum(when(~col("old_match") &  col("new_match"), 1).otherwise(0)).alias("FP"),
    spark_sum(when(~col("old_match") & ~col("new_match"), 1).otherwise(0)).alias("TN"),
).collect()[0]

TP, FN, FP, TN = counts["TP"], counts["FN"], counts["FP"], counts["TN"]
total = TP + FN + FP + TN

precision = TP / (TP + FP) if (TP + FP) > 0 else 0.0
recall = TP / (TP + FN) if (TP + FN) > 0 else 0.0
f1 = 2 * precision * recall / (precision + recall) if (precision + recall) > 0 else 0.0

print("=" * 70)
print("OLD vs NEW CLUSTER LOGIC  (id_i==id_j  vs  cluster_id_i==cluster_id_j)")
print("=" * 70)
print(f"  Total pairs                         : {total:>10,}")
print(f"  TP  old=match, new=match            : {TP:>10,}  ← both agree: match")
print(f"  FN  old=match, new=miss             : {FN:>10,}  ← new lost a match")
print(f"  FP  old=miss,  new=match            : {FP:>10,}  ← new gained a match")
print(f"  TN  old=miss,  new=miss             : {TN:>10,}  ← both agree: no match")
print()
print(f"  Precision : {precision:.4f}  (of new matches, how many were also old matches)")
print(f"  Recall    : {recall:.4f}  (of old matches, how many did new also catch)")
print(f"  F1        : {f1:.4f}")

# ── Build FP / FN DataFrames and write to CSV ─────────────────────────────────

# Lead columns to put first in the CSV for easy manual review
LEAD_COLS = [
    "id_i", "id_j", "uid_i", "uid_j",
    "cluster_id_i", "cluster_id_j",
    "providerName_i", "providerName_j",
    "name_i", "name_j",
    "combined_address_i", "combined_address_j",
    "geoCode_lat_i", "geoCode_lat_j",
    "geoCode_long_i", "geoCode_long_j",
    "starRating_i", "starRating_j",
    "overall_pair_score",
]

def ordered_cols(df):
    lead = [c for c in LEAD_COLS if c in df.columns]
    rest = [c for c in df.columns if c not in set(lead)]
    return lead + rest

fp_df = df.filter(~col("old_match") &  col("new_match")).drop("old_match", "new_match")
fn_df = df.filter(col("old_match") & ~col("new_match")).drop("old_match", "new_match")

fp_pdf = fp_df.select(ordered_cols(fp_df)).toPandas()
fn_pdf = fn_df.select(ordered_cols(fn_df)).toPandas()

fp_pdf.to_csv(FP_CSV, index=False)

fn_pdf.to_csv(FN_CSV, index=False)

print(f"  ✓ FN ({FN:,} rows) → {FN_CSV}")
print()
print(f"  ✓ FP ({FP:,} rows) → {FP_CSV}")

OLD vs NEW CLUSTER LOGIC  (id_i==id_j  vs  cluster_id_i==cluster_id_j)
  Total pairs                         :    233,476
  TP  old=match, new=match            :     37,317  ← both agree: match
  FN  old=match, new=miss             :      3,900  ← new lost a match
  FP  old=miss,  new=match            :        301  ← new gained a match
  TN  old=miss,  new=miss             :    191,958  ← both agree: no match

  Precision : 0.9920  (of new matches, how many were also old matches)
  Recall    : 0.9054  (of old matches, how many did new also catch)
  F1        : 0.9467


  ✓ FN (3,900 rows) → /Users/nakul.patil/Downloads/hotel_mapping_reports/cluster_fn_20260314_212151.csv

  ✓ FP (301 rows) → /Users/nakul.patil/Downloads/hotel_mapping_reports/cluster_fp_20260314_212151.csv


In [8]:

# cell 8 — FN Condition Analysis
# FN pairs: old logic matched them (id_i == id_j) but new clustering put them in different clusters.
# Since the new system missed them, at least one AND-segment of the WHERE clause must have failed.
#
# Margin columns show a value ONLY when:
#   1. the AND-segment (group) the condition belongs to is FAILING, AND
#   2. the margin itself is negative (this specific condition failed)
#
# SHOW_CLOSEST_FAILED_SCORE = True  (recommended)
#   → per failing group, show ONLY the single score that was closest to its threshold
#     (highest / least-negative margin). This immediately tells you which signal to tune.
# SHOW_CLOSEST_FAILED_SCORE = False
#   → show all negative margins from every failing group.
#
# SENSITIVITY_PCTS — list of % threshold relaxations to test in the near-miss section.
#   For each signal, counts how many FN pairs would be resolved (full WHERE clause passes)
#   if that ONE threshold is relaxed by each %.
#
# Group-pass status columns show whether each AND-segment passed overall.
# where_clause_pass shows if the full WHERE clause matched (always False for FN pairs).

import yaml
from collections import Counter
from pyspark.sql.functions import (
    col, lit, abs as spark_abs, round as spark_round,
    expr, when, concat_ws, greatest,
)

CONFIG_PATH     = "/Users/nakul.patil/Documents/hotel-mapping/src/hotel_data/config/config.yaml"

REPORTS_DIR = os.path.expanduser("~/Downloads/hotel_mapping_reports")
os.makedirs(REPORTS_DIR, exist_ok=True)
if "RUN_TIMESTAMP" not in globals():
    RUN_TIMESTAMP = datetime.datetime.now().strftime("%Y%m%d_%H%M%S")

FN_ANALYSIS_CSV = os.path.join(REPORTS_DIR, f"fn_condition_analysis_{RUN_TIMESTAMP}.csv")

# ── Config ────────────────────────────────────────────────────────────────────
# True  → per failing group keep only the score closest to its threshold (easiest to fix)
# False → show every negative margin in every failing group
SHOW_CLOSEST_FAILED_SCORE = True

# % threshold relaxations to evaluate in the near-miss sensitivity section
SENSITIVITY_PCTS = [5, 10, 15, 20]

# Human-readable names for each AND-segment (positionally matched to match_logic.rules)
GROUP_NAMES = [
    "name_closeness",
    "address_distance_closeness",
    "property_type_closeness",
    "sub_property_closeness",
    "house_number_closeness",
]

CMP_SYM = {"gte": ">=", "lte": "<=", "gt": ">", "lt": "<", "eq": "=", "neq": "!="}

def flatten_leaves(node):
    if "signal" in node:
        return [node]
    out = []
    for r in node["rules"]:
        out.extend(flatten_leaves(r))
    return out

def to_spark_expr_str(node):
    if "signal" in node:
        cmp = CMP_SYM.get(node["comparator"], node["comparator"])
        return f"({node['signal']} {cmp} {node['threshold']})"
    op    = f" {node['operator'].upper()} "
    parts = [to_spark_expr_str(r) for r in node["rules"]]
    return "(" + op.join(parts) + ")"

def to_spark_expr_str_with_override(node, override_key, new_thr):
    """Build a WHERE-clause expression string with ONE threshold replaced.
    override_key = (signal, comparator_str, float(original_threshold))
    """
    if "signal" in node:
        cmp_sym = CMP_SYM.get(node["comparator"], node["comparator"])
        key = (node["signal"], node["comparator"], float(node["threshold"]))
        thr = new_thr if key == override_key else node["threshold"]
        return f"({node['signal']} {cmp_sym} {thr})"
    op = f" {node['operator'].upper()} "
    return "(" + op.join(
        to_spark_expr_str_with_override(r, override_key, new_thr)
        for r in node["rules"]
    ) + ")"

with open(CONFIG_PATH) as f:
    cfg = yaml.safe_load(f)
match_logic = cfg["scoring"]["match_logic"]

group_nodes = match_logic["rules"] if match_logic.get("operator") == "AND" else [match_logic]

group_labels = [
    GROUP_NAMES[i] if i < len(GROUP_NAMES) else f"group_{i}"
    for i in range(len(group_nodes))
]

print(f"AND segments: {len(group_nodes)}")
for label, gnode in zip(group_labels, group_nodes):
    op     = gnode.get("operator", "LEAF")
    leaves = flatten_leaves(gnode)
    sigs   = [f"{l['signal']} {CMP_SYM.get(l['comparator'],l['comparator'])} {l['threshold']}"
              for l in leaves]
    print(f"  {label} [{op}]: {sigs}")
print(f"\nSHOW_CLOSEST_FAILED_SCORE = {SHOW_CLOSEST_FAILED_SCORE}\n")

# ── FN pairs ──────────────────────────────────────────────────────────────────
fn_df = (
    hotel_pairs
    .withColumn("old_match", col("id_i") == col("id_j"))
    .withColumn("new_match", col("cluster_id_i") == col("cluster_id_j"))
    .filter(col("old_match") & ~col("new_match"))
    .drop("old_match", "new_match")
)
fn_df = fn_df.cache()          # cache — reused many times in sensitivity analysis
fn_total = fn_df.count()
print(f"FN pairs to analyse: {fn_total:,}\n")

# ── Detect (signal, threshold) that span multiple AND-segments ────────────────
sig_thr_count = Counter()
for gnode in group_nodes:
    for leaf in flatten_leaves(gnode):
        thr_str = str(leaf["threshold"]).rstrip("0").rstrip(".")
        sig_thr_count[(leaf["signal"], thr_str)] += 1

# ── Group status expressions ──────────────────────────────────────────────────
group_pass_col_names = [f"{label}_pass" for label in group_labels]
group_status_exprs = [
    expr(to_spark_expr_str(gnode)).alias(pass_col)
    for pass_col, gnode in zip(group_pass_col_names, group_nodes)
]
overall_pass_expr = expr(to_spark_expr_str(match_logic)).alias("where_clause_pass")

# ── Margin expressions ────────────────────────────────────────────────────────
# When SHOW_CLOSEST_FAILED_SCORE=True:
#   For each failing AND-segment, compute the raw margin for every leaf.
#   Mask passing margins to -∞, then take greatest() → the least-negative (closest to threshold).
#   Show a leaf's margin only if it equals that closest value (and group fails and margin < 0).
# When False: show all negative margins in failing groups (as before).

margin_exprs     = []
margin_col_names = []
skipped          = []

NEG_INF = float("-inf")

for label, group_node in zip(group_labels, group_nodes):
    group_pass  = expr(to_spark_expr_str(group_node))
    group_leaves = [
        leaf for leaf in flatten_leaves(group_node)
        if leaf["signal"] in fn_df.columns
    ]
    # Track signals to skip that aren't in the dataframe
    for leaf in flatten_leaves(group_node):
        if leaf["signal"] not in fn_df.columns and leaf["signal"] not in skipped:
            skipped.append(leaf["signal"])

    if not group_leaves:
        continue

    # Build raw rounded margin for each leaf in this group
    leaf_rounded = {}
    for leaf in group_leaves:
        sig, cmp, thr = leaf["signal"], leaf["comparator"], leaf["threshold"]
        thr_f = float(thr)
        if cmp in ("gte", "gt"):
            raw = col(sig) - lit(thr_f)
        elif cmp in ("lte", "lt"):
            raw = lit(thr_f) - col(sig)
        else:
            raw = spark_abs(col(sig) - lit(thr_f))
        leaf_rounded[sig] = spark_round(raw, 2)

    if SHOW_CLOSEST_FAILED_SCORE:
        # For each leaf: mask to -∞ if margin ≥ 0 (condition passed), keep otherwise.
        # greatest() across masked values → the closest-to-threshold failing margin.
        masked = [
            when(r < lit(0), r).otherwise(lit(NEG_INF))
            for r in leaf_rounded.values()
        ]
        closest_failing = greatest(*masked) if len(masked) > 1 else masked[0]

    for leaf in group_leaves:
        sig, cmp, thr = leaf["signal"], leaf["comparator"], leaf["threshold"]
        thr_str  = str(thr).rstrip("0").rstrip(".")
        suffix   = f"_{label}" if sig_thr_count[(sig, thr_str)] > 1 else ""
        col_name = f"{sig}_margin_{thr_str}{suffix}"

        if col_name in margin_col_names:
            continue

        rounded = leaf_rounded[sig]

        if SHOW_CLOSEST_FAILED_SCORE:
            # Show only: group failed AND margin < 0 AND this is the closest-to-threshold one
            margin_expr = (
                when(group_pass | (rounded >= lit(0)) | (rounded != closest_failing), lit(None))
                .otherwise(rounded)
            )
        else:
            # Show all: group failed AND margin < 0
            margin_expr = (
                when(group_pass | (rounded >= lit(0)), lit(None))
                .otherwise(rounded)
            )

        margin_exprs.append(margin_expr.alias(col_name))
        margin_col_names.append(col_name)

if skipped:
    print(f"Note: signals not found in hotel_pairs, skipped: {skipped}")

# ── Lead identity / context columns ──────────────────────────────────────────
LEAD_COLS = [
    "id_i", "id_j", "uid_i", "uid_j",
    "cluster_id_i", "cluster_id_j",
    "providerName_i", "providerName_j",
    "name_i", "name_j",
    "combined_address_i", "combined_address_j",
    "starRating_i", "starRating_j",
    "overall_pair_score",
    "providerHotelId_i", "providerHotelId_j",
    "type_i", "type_j",
    "geo_distance_km",
]
lead_present = [c for c in LEAD_COLS if c in fn_df.columns]

fn_final = fn_df.select(
    *[col(c) for c in lead_present],
    overall_pass_expr,
    *group_status_exprs,
    *margin_exprs,
)

print("Sample (truncated to 40 chars):")
fn_final.show(10, truncate=40)

# ── Write to CSV ──────────────────────────────────────────────────────────────
fn_pdf = fn_final.toPandas()
fn_pdf.to_csv(FN_ANALYSIS_CSV, index=False)
print(f"\n✓ FN condition analysis ({len(fn_pdf):,} rows, {len(fn_pdf.columns)} cols) → {FN_ANALYSIS_CSV}")
print(f"  Status columns : where_clause_pass, " + ", ".join(group_pass_col_names))
print(f"  Margin columns ({len(margin_col_names)}): {margin_col_names}")

# ── Failure attribution stats ─────────────────────────────────────────────────
print()
print("=" * 70)
print("FAILURE ATTRIBUTION  —  which group(s) caused each FN miss")
print("=" * 70)
print("  Interpretation:")
print("  • Few groups failing  → threshold tuning likely fixes it (real FN)")
print("  • Many groups failing → pair looks genuinely different; ground-truth")
print("    label (id_i == id_j) may be incorrect rather than the new clustering.")
print()

fail_label_expr = concat_ws(" + ", *[
    when(~col(pc), lit(label)).otherwise(lit(None))
    for label, pc in zip(group_labels, group_pass_col_names)
]).alias("failing_groups")

stats_df = fn_final.select(fail_label_expr).toPandas()
stats_df["failing_groups"] = stats_df["failing_groups"].replace("", "none (where_clause passed?)")

combo_counts = stats_df["failing_groups"].value_counts().reset_index()
combo_counts.columns = ["failing_groups", "count"]
combo_counts["pct"] = (combo_counts["count"] / fn_total * 100).round(1)
combo_counts["num_groups_failing"] = combo_counts["failing_groups"].apply(
    lambda s: len(s.split(" + ")) if s != "none (where_clause passed?)" else 0
)
combo_counts = combo_counts.sort_values("count", ascending=False).reset_index(drop=True)

print(f"  {'Failing group combination':<55} {'Count':>6}  {'%':>5}  {'#groups':>7}")
print(f"  {'-'*55} {'-'*6}  {'-'*5}  {'-'*7}")
for _, row in combo_counts.iterrows():
    suspicion = "  ← possible bad ground-truth" if row["num_groups_failing"] >= 3 else ""
    print(f"  {row['failing_groups']:<55} {row['count']:>6,}  {row['pct']:>4.1f}%  {row['num_groups_failing']:>7}{suspicion}")

print()
bad_gt_count = combo_counts.loc[combo_counts["num_groups_failing"] >= 3, "count"].sum()
bad_gt_pct   = bad_gt_count / fn_total * 100 if fn_total > 0 else 0
print(f"  Rows with ≥3 groups failing (likely bad ground-truth): {bad_gt_count:,}  ({bad_gt_pct:.1f}%)")
print(f"  Rows with 1-2 groups failing (tunable thresholds)    : {fn_total - bad_gt_count:,}  ({100 - bad_gt_pct:.1f}%)")

# ── Closest-signal frequency (only when SHOW_CLOSEST_FAILED_SCORE=True) ───────
if SHOW_CLOSEST_FAILED_SCORE:
    print()
    print("=" * 70)
    print("CLOSEST SIGNAL FREQUENCY  —  which score to tune per failing group")
    print("  (the non-NULL column per row = the signal closest to its threshold)")
    print("=" * 70)
    for mc in margin_col_names:
        non_null = fn_pdf[mc].notna().sum()
        pct      = non_null / fn_total * 100 if fn_total > 0 else 0
        print(f"  {mc:<55} {non_null:>6,}  ({pct:>4.1f}%)")

# ── Threshold sensitivity  (near-miss analysis) ───────────────────────────────
# For each unique signal/threshold in the WHERE clause, relax it by SENSITIVITY_PCTS
# and count how many FN pairs would be resolved (the full WHERE clause now passes).
# Only one threshold is changed at a time — combinations are not tested.
#
# Relax direction:
#   gte / gt  → reduce threshold  (e.g. name_score >= 0.80 → >= 0.72 at -10%)
#   lte / lt  → increase threshold (e.g. geo_distance_km <= 50 → <= 55 at +10%)

print()
print("=" * 70)
print("THRESHOLD SENSITIVITY  —  near-miss FN analysis")
print(f"  Base: {fn_total:,} FN pairs.")
print(f"  Each cell = FN pairs resolved (WHERE clause now passes) if ONLY that")
print(f"  one threshold is relaxed by that %  (gte/gt: reduced; lte/lt: increased).")
print()

# Collect unique (signal, comparator, threshold) leaves present in the dataframe
seen_sens: set = set()
sens_leaves = []
for gnode in group_nodes:
    for leaf in flatten_leaves(gnode):
        key = (leaf["signal"], leaf["comparator"], float(leaf["threshold"]))
        if key not in seen_sens and leaf["signal"] in fn_df.columns:
            seen_sens.add(key)
            sens_leaves.append((leaf, key))

CW = 15  # column width for each % cell
pct_header = "  ".join(f"-{p}%".center(CW) for p in SENSITIVITY_PCTS)
print(f"  {'Signal / threshold':<52}  {pct_header}")
print(f"  {'-'*52}  " + "  ".join(["-" * CW] * len(SENSITIVITY_PCTS)))

best_label, best_relaxed_pct, best_resolved = "", 0, 0

for leaf, sens_key in sens_leaves:
    sig, cmp_key, thr = leaf["signal"], leaf["comparator"], float(leaf["threshold"])
    cmp_sym = CMP_SYM.get(cmp_key, cmp_key)
    label   = f"{sig} {cmp_sym} {thr}"
    cells   = []

    for pct in SENSITIVITY_PCTS:
        factor = pct / 100.0
        if cmp_key in ("gte", "gt"):
            new_thr = round(thr * (1 - factor), 6)
        elif cmp_key in ("lte", "lt"):
            new_thr = round(thr * (1 + factor), 6)
        else:
            cells.append("N/A".center(CW))
            continue

        new_where_expr = to_spark_expr_str_with_override(match_logic, sens_key, new_thr)
        resolved = fn_df.filter(expr(new_where_expr)).count()
        pct_val  = resolved / fn_total * 100 if fn_total > 0 else 0

        if resolved > best_resolved:
            best_resolved, best_label, best_relaxed_pct = resolved, label, pct

        cells.append(f"{resolved} ({pct_val:.1f}%)".center(CW))

    print(f"  {label:<52}  " + "  ".join(cells))

print()

if best_label:

    best_pct_of_fn = best_resolved / fn_total * 100 if fn_total > 0 else 0
    print(f"  >>> Most impactful single change:")
    print(f"      Relax '{best_label}' by -{best_relaxed_pct}%")
    print(f"      → resolves {best_resolved:,} / {fn_total:,} FN pairs  ({best_pct_of_fn:.1f}%)")

AND segments: 6
  name_closeness [OR]: ['name_score_jaccard >= 0.9', 'name_score_lcs >= 0.9', 'name_score_sbert >= 0.9', 'name_score_levenshtein >= 0.95', 'normalized_name_score_jaccard >= 0.95', 'normalized_name_score_lcs >= 0.95', 'normalized_name_score_levenshtein >= 0.95', 'normalized_name_score_sbert >= 0.95', 'average_name_score >= 0.8', 'average_normalized_name_score >= 0.9', 'normalized_name_score_containment >= 1', 'name_score_sbert >= 0.75', 'geo_distance_km <= 0.2', 'normalized_name_score_containment >= 1', 'name_score_sbert >= 0.7', 'geo_distance_km <= 0.1', 'name_score_containment >= 0.95', 'normalized_name_score_containment >= 0.95', 'star_ratings_score >= 0.8', 'geo_distance_km <= 0.2']
  address_distance_closeness [OR]: ['address_line1_score >= 0.5', 'address_sbert_score >= 0.5', 'geo_distance_km <= 0.2']
  property_type_closeness [LEAF]: ['property_type_score >= 0.5']
  sub_property_closeness [LEAF]: ['name_unit_score >= 0.5']
  house_number_closeness [OR]: ['address_u

FN pairs to analyse: 3,900

Sample (truncated to 40 chars):
+--------+--------+----------------------------------------+----------------------------------------+------------+------------+--------------+--------------+----------------------------------------+----------------------------------------+----------------------------------------+----------------------------------------+------------+------------+------------------+-----------------+-----------------+---------------+---------------+---------------------+-----------------+-------------------+-------------------------------+----------------------------+---------------------------+---------------------------+------------+-----------------------------+-------------------------+---------------------------+----------------------------------+-----------------------------------------+-------------------------------------+---------------------------------------------+---------------------------------------+-----------------------------+-

In [9]:

# cell 9 — FP Condition Analysis
# FP pairs: new clustering matched them (cluster_id_i == cluster_id_j) but old system didn't (id_i != id_j).
# ALL AND-segments of the WHERE clause PASS for FP pairs — that's why they got matched.
#
# Margin = gap ABOVE threshold (positive = passes, close to 0 = barely passed).
# We want to find the scores that are just barely above their threshold —
# lowering that threshold (or improving the underlying scoring) would reject this FP.
#
# SHOW_CLOSEST_PASS_SCORE = True  (recommended)
#   → per passing group, show ONLY the score closest to its threshold from above
#     (lowest/least-positive margin). Most fragile condition — easiest to tighten.
# SHOW_CLOSEST_PASS_SCORE = False
#   → show all positive margins from every passing group.
#
# FAILURE ATTRIBUTION — for each FP pair, which group had the narrowest margin?
#   is_single_or_pass: True if exactly 1 leaf was passing in that OR-group.
#   → a single-pass group means tightening just that 1 signal rejects the pair.
#   → a multi-pass group means ALL passing leaves need tightening.
#
# THRESHOLD SENSITIVITY — % of FP pairs eliminated when ONE threshold is tightened by X%.
#   gte/gt: threshold raised; lte/lt: threshold lowered.

import yaml
import pandas as pd
from collections import Counter
from functools import reduce as _reduce
from pyspark.sql.functions import (
    col, lit, abs as spark_abs, round as spark_round,
    expr, when, least,
)

CONFIG_PATH     = "/Users/nakul.patil/Documents/hotel-mapping/src/hotel_data/config/config.yaml"

REPORTS_DIR = os.path.expanduser("~/Downloads/hotel_mapping_reports")
os.makedirs(REPORTS_DIR, exist_ok=True)
if "RUN_TIMESTAMP" not in globals():
    RUN_TIMESTAMP = datetime.datetime.now().strftime("%Y%m%d_%H%M%S")

FP_ANALYSIS_CSV = os.path.join(REPORTS_DIR, f"fp_condition_analysis_{RUN_TIMESTAMP}.csv")

# ── Config ────────────────────────────────────────────────────────────────────
# True  → per passing group keep only the score closest to its threshold from above
# False → show every positive margin in every passing group
SHOW_CLOSEST_PASS_SCORE = True

# % threshold tightenings to evaluate in the near-miss sensitivity section
SENSITIVITY_PCTS = [5, 10, 15, 20]

# Human-readable names for each AND-segment (positionally matched to match_logic.rules)
GROUP_NAMES = [
    "name_closeness",
    "address_distance_closeness",
    "property_type_closeness",
    "sub_property_closeness",
    "house_number_closeness",
]

CMP_SYM = {"gte": ">=", "lte": "<=", "gt": ">", "lt": "<", "eq": "=", "neq": "!="}

def flatten_leaves(node):
    if "signal" in node:
        return [node]
    out = []
    for r in node["rules"]:
        out.extend(flatten_leaves(r))
    return out

def to_spark_expr_str(node):
    if "signal" in node:
        cmp = CMP_SYM.get(node["comparator"], node["comparator"])
        return f"({node['signal']} {cmp} {node['threshold']})"
    op = f" {node['operator'].upper()} "
    return "(" + op.join(to_spark_expr_str(r) for r in node["rules"]) + ")"

def to_spark_expr_str_with_override(node, override_key, new_thr):
    """Rebuild the WHERE clause expression with exactly ONE threshold replaced.
    override_key = (signal, comparator_str, float(original_threshold))
    """
    if "signal" in node:
        cmp_sym = CMP_SYM.get(node["comparator"], node["comparator"])
        key = (node["signal"], node["comparator"], float(node["threshold"]))
        thr = new_thr if key == override_key else node["threshold"]
        return f"({node['signal']} {cmp_sym} {thr})"
    op = f" {node['operator'].upper()} "
    return "(" + op.join(
        to_spark_expr_str_with_override(r, override_key, new_thr)
        for r in node["rules"]
    ) + ")"

with open(CONFIG_PATH) as f:
    cfg = yaml.safe_load(f)
match_logic = cfg["scoring"]["match_logic"]

group_nodes = match_logic["rules"] if match_logic.get("operator") == "AND" else [match_logic]
group_labels = [
    GROUP_NAMES[i] if i < len(GROUP_NAMES) else f"group_{i}"
    for i in range(len(group_nodes))
]

print(f"AND segments: {len(group_nodes)}")
for label, gnode in zip(group_labels, group_nodes):
    op     = gnode.get("operator", "LEAF")
    leaves = flatten_leaves(gnode)
    sigs   = [f"{l['signal']} {CMP_SYM.get(l['comparator'], l['comparator'])} {l['threshold']}"
              for l in leaves]
    print(f"  {label} [{op}]: {sigs}")
print(f"\nSHOW_CLOSEST_PASS_SCORE = {SHOW_CLOSEST_PASS_SCORE}\n")

# ── FP pairs ──────────────────────────────────────────────────────────────────
fp_df = (
    hotel_pairs
    .withColumn("old_match", col("id_i") == col("id_j"))
    .withColumn("new_match", col("cluster_id_i") == col("cluster_id_j"))
    .filter(~col("old_match") & col("new_match"))
    .drop("old_match", "new_match")
)
fp_df = fp_df.cache()           # cache — reused many times in sensitivity analysis
fp_total = fp_df.count()
print(f"FP pairs to analyse: {fp_total:,}\n")

# ── Signal/threshold collision detection  ─────────────────────────────────────
sig_thr_count = Counter()
for gnode in group_nodes:
    for leaf in flatten_leaves(gnode):
        thr_str = str(leaf["threshold"]).rstrip("0").rstrip(".")
        sig_thr_count[(leaf["signal"], thr_str)] += 1

# ── Group status expressions ──────────────────────────────────────────────────
group_pass_col_names = [f"{label}_pass" for label in group_labels]
group_status_exprs = [
    expr(to_spark_expr_str(gnode)).alias(pass_col)
    for pass_col, gnode in zip(group_pass_col_names, group_nodes)
]
overall_pass_expr = expr(to_spark_expr_str(match_logic)).alias("where_clause_pass")

# ── Margin + n_pass expressions ───────────────────────────────────────────────
# Margin = gap ABOVE threshold:
#   gte/gt: score - threshold  → positive means passes
#   lte/lt: threshold - score  → positive means passes
#
# Show margin only when: group passes AND margin >= 0
#   AND (if SHOW_CLOSEST_PASS_SCORE) margin == closest_passing for that group
#
# closest_passing per group:
#   mask each leaf to +INF if margin < 0 (leaf failed), then least() = smallest positive
#   = the leaf that BARELY passed = most actionable to tighten
#
# n_pass = count of leaves in a group with margin >= 0.
#   n_pass == 1 → single OR pass → tighten 1 signal to reject the pair
#   n_pass >  1 → multi  OR pass → must tighten all passing signals

margin_exprs     = []
margin_col_names = []
n_pass_exprs     = []
n_pass_col_names = []
col_to_group     = {}   # margin_col_name → group_label  (used for attribution)
skipped          = []

POS_INF = float("inf")

for label, group_node in zip(group_labels, group_nodes):
    group_pass = expr(to_spark_expr_str(group_node))

    # Partition leaves: computable (gte/gt/lte/lt present in df) vs skipped
    all_leaves = flatten_leaves(group_node)
    computable_leaves = [
        l for l in all_leaves
        if l["signal"] in fp_df.columns and l["comparator"] in ("gte", "gt", "lte", "lt")
    ]
    for l in all_leaves:
        if l["signal"] not in fp_df.columns and l["signal"] not in skipped:
            skipped.append(l["signal"])

    if not computable_leaves:
        continue

    # Raw rounded margin for each computable leaf in this group
    leaf_rounded = {}
    for leaf in computable_leaves:
        sig, cmp, thr = leaf["signal"], leaf["comparator"], leaf["threshold"]
        thr_f = float(thr)
        if cmp in ("gte", "gt"):
            raw = col(sig) - lit(thr_f)
        else:   # lte / lt
            raw = lit(thr_f) - col(sig)
        leaf_rounded[sig] = spark_round(raw, 2)

    # n_pass: count of leaves with margin >= 0 in this group
    n_pass_col = _reduce(
        lambda a, b: a + b,
        [when(r >= lit(0), lit(1)).otherwise(lit(0)) for r in leaf_rounded.values()]
    ).alias(f"{label}_n_pass")
    n_pass_exprs.append(n_pass_col)
    n_pass_col_names.append(f"{label}_n_pass")

    if SHOW_CLOSEST_PASS_SCORE:
        # Mask failed margins to +INF, take least() → smallest positive = closest to threshold
        masked_positive = [
            when(r >= lit(0), r).otherwise(lit(POS_INF))
            for r in leaf_rounded.values()
        ]
        closest_passing = least(*masked_positive) if len(masked_positive) > 1 else masked_positive[0]

    for leaf in computable_leaves:
        sig, cmp, thr = leaf["signal"], leaf["comparator"], leaf["threshold"]
        if sig not in leaf_rounded:
            continue
        thr_str  = str(thr).rstrip("0").rstrip(".")
        suffix   = f"_{label}" if sig_thr_count[(sig, thr_str)] > 1 else ""
        col_name = f"{sig}_margin_{thr_str}{suffix}"

        if col_name in margin_col_names:
            continue

        rounded = leaf_rounded[sig]

        if SHOW_CLOSEST_PASS_SCORE:
            # Show only: group passes AND margin >= 0 AND this is the closest-to-threshold one
            margin_expr = (
                when(~group_pass | (rounded < lit(0)) | (rounded != closest_passing), lit(None))
                .otherwise(rounded)
            )
        else:
            # Show all: group passes AND margin >= 0
            margin_expr = (
                when(~group_pass | (rounded < lit(0)), lit(None))
                .otherwise(rounded)
            )

        margin_exprs.append(margin_expr.alias(col_name))
        margin_col_names.append(col_name)
        col_to_group[col_name] = label

if skipped:
    print(f"Note: signals not found in hotel_pairs, skipped: {skipped}")

# ── Lead identity / context columns ──────────────────────────────────────────
LEAD_COLS = [
    "id_i", "id_j", "uid_i", "uid_j",
    "cluster_id_i", "cluster_id_j",
    "providerName_i", "providerName_j",
    "name_i", "name_j",
    "combined_address_i", "combined_address_j",
    "starRating_i", "starRating_j",
    "overall_pair_score",
    "providerHotelId_i", "providerHotelId_j",
    "type_i", "type_j",
    "geo_distance_km",
]
lead_present = [c for c in LEAD_COLS if c in fp_df.columns]

fp_final = fp_df.select(
    *[col(c) for c in lead_present],
    overall_pass_expr,
    *group_status_exprs,
    *margin_exprs,
    *n_pass_exprs,
)

print("Sample (truncated to 40 chars):")
fp_final.show(10, truncate=40)

# ── Write to CSV ──────────────────────────────────────────────────────────────
fp_pdf = fp_final.toPandas()
fp_pdf.to_csv(FP_ANALYSIS_CSV, index=False)
print(f"\n✓ FP condition analysis ({len(fp_pdf):,} rows, {len(fp_pdf.columns)} cols) → {FP_ANALYSIS_CSV}")
print(f"  Status columns : where_clause_pass, " + ", ".join(group_pass_col_names))
print(f"  Margin columns ({len(margin_col_names)}): {margin_col_names}")
print(f"  n_pass columns : {n_pass_col_names}")

# ── Failure attribution stats ─────────────────────────────────────────────────
print()
print("=" * 70)
print("FAILURE ATTRIBUTION  —  most fragile group per FP pair")
print("=" * 70)
print("  For each FP pair: which AND-group had the smallest positive margin?")
print("  → that group is the single weakest link — tighten it to reject the pair.")
print("  1-leaf pass: only 1 OR-condition was passing in that group.")
print("  → if 1-leaf pass, tightening just THAT signal rejects this FP.")
print("  → if multi-pass, all passing leaves in the group need tightening.")
print()

if margin_col_names:
    margin_df = fp_pdf[margin_col_names].copy()

    # Most fragile: the margin column with the smallest positive value per row
    most_fragile_col    = margin_df.idxmin(axis=1)   # NaN when no non-null values
    most_fragile_margin = margin_df.min(axis=1)

    fp_pdf2 = fp_pdf[margin_col_names + n_pass_col_names].copy()
    fp_pdf2["most_fragile_col"]    = most_fragile_col
    fp_pdf2["most_fragile_margin"] = most_fragile_margin
    fp_pdf2["most_fragile_group"]  = fp_pdf2["most_fragile_col"].map(
        lambda c: col_to_group.get(c, "unknown") if isinstance(c, str) else "no_data"
    )

    def _get_n_pass(row):
        grp = row["most_fragile_group"]
        if grp in ("unknown", "no_data"):
            return None
        return row.get(f"{grp}_n_pass", None)

    fp_pdf2["most_fragile_n_pass"] = fp_pdf2.apply(_get_n_pass, axis=1)
    fp_pdf2["is_single_or_pass"]   = fp_pdf2["most_fragile_n_pass"] == 1

    table_data = fp_pdf2.groupby(["most_fragile_col", "most_fragile_group"]).agg(
        count        =("most_fragile_margin", "count"),
        avg_margin   =("most_fragile_margin", "mean"),
        min_margin   =("most_fragile_margin", "min"),
        single_count =("is_single_or_pass",   "sum"),
    ).reset_index()
    table_data["pct"]        = (table_data["count"] / fp_total * 100).round(1)
    table_data["single_pct"] = (table_data["single_count"] / table_data["count"] * 100).round(0).astype(int)
    table_data = table_data.sort_values("count", ascending=False).reset_index(drop=True)

    print(f"  {'Signal (most fragile in its group)':<50}  {'Group':<28}  {'1-leaf%':>7}  {'Count':>6}  {'%':>5}  {'Avg margin':>10}")
    print(f"  {'-'*50}  {'-'*28}  {'-'*7}  {'-'*6}  {'-'*5}  {'-'*10}")
    for _, row in table_data.iterrows():
        action = "  ← tune 1 signal" if row["single_pct"] >= 70 else ""
        print(
            f"  {row['most_fragile_col']:<50}  "
            f"{row['most_fragile_group']:<28}  "
            f"{row['single_pct']:>6}%  "
            f"{row['count']:>6,}  "
            f"{row['pct']:>4.1f}%  "
            f"{row['avg_margin']:>10.4f}"
            f"{action}"
        )

    print()
    single_pass_rows = int(fp_pdf2["is_single_or_pass"].sum())
    multi_pass_rows  = fp_total - single_pass_rows
    print(f"  Pairs where 1 signal fix suffices (single OR pass) : {single_pass_rows:,}  ({single_pass_rows/fp_total*100:.1f}%)")
    print(f"  Pairs needing multiple signal fixes (multi  OR pass): {multi_pass_rows:,}  ({multi_pass_rows/fp_total*100:.1f}%)")
else:
    print("  No margin columns available for attribution.")

# ── Closest signal frequency ──────────────────────────────────────────────────
if SHOW_CLOSEST_PASS_SCORE and margin_col_names:
    print()
    print("=" * 70)
    print("CLOSEST SIGNAL FREQUENCY  —  which score to tighten per passing group")
    print("  (the non-NULL column per row = the signal barely above its threshold)")
    print("=" * 70)
    for mc in margin_col_names:
        non_null = fp_pdf[mc].notna().sum()
        pct      = non_null / fp_total * 100 if fp_total > 0 else 0
        print(f"  {mc:<55} {non_null:>6,}  ({pct:>4.1f}%)")

# ── Threshold sensitivity  (tighten thresholds → eliminate FPs) ───────────────
# For each signal/threshold, TIGHTEN by SENSITIVITY_PCTS and count FPs eliminated.
#   gte/gt: increase threshold (harder to pass e.g. name_score >=0.80 → >=0.88 at +10%)
#   lte/lt: decrease threshold (harder to pass e.g. geo_dist <= 50 → <= 45 at -10%)
# Count = FP pairs eliminated (WHERE clause now FAILS with the tighter threshold).

print()
print("=" * 70)
print("THRESHOLD SENSITIVITY  —  near-miss FP analysis")
print(f"  Base: {fp_total:,} FP pairs.")
print(f"  Each cell = FP pairs ELIMINATED (WHERE clause fails) if ONLY that")
print(f"  one threshold is tightened by that %  (gte/gt: raised; lte/lt: lowered).")
print()

seen_sens: set = set()
sens_leaves = []
for gnode in group_nodes:
    for leaf in flatten_leaves(gnode):
        key = (leaf["signal"], leaf["comparator"], float(leaf["threshold"]))
        if (
            key not in seen_sens
            and leaf["signal"] in fp_df.columns
            and leaf["comparator"] in ("gte", "gt", "lte", "lt")
        ):
            seen_sens.add(key)
            sens_leaves.append((leaf, key))

CW = 17  # column width
pct_header = "  ".join(f"+{p}% stricter".center(CW) for p in SENSITIVITY_PCTS)
print(f"  {'Signal / threshold':<52}  {pct_header}")
print(f"  {'-'*52}  " + "  ".join(["-" * CW] * len(SENSITIVITY_PCTS)))

best_label, best_tightened_pct, best_eliminated = "", 0, 0

for leaf, sens_key in sens_leaves:
    sig, cmp_key, thr = leaf["signal"], leaf["comparator"], float(leaf["threshold"])
    cmp_sym = CMP_SYM.get(cmp_key, cmp_key)
    label   = f"{sig} {cmp_sym} {thr}"
    cells   = []

    for pct in SENSITIVITY_PCTS:
        factor = pct / 100.0
        if cmp_key in ("gte", "gt"):
            new_thr = round(thr * (1 + factor), 6)   # raise = stricter
        else:   # lte / lt
            new_thr = round(thr * (1 - factor), 6)   # lower = stricter

        new_where_expr = to_spark_expr_str_with_override(match_logic, sens_key, new_thr)
        # Count FP pairs that would be ELIMINATED (WHERE clause now fails)
        eliminated = fp_df.filter(~expr(new_where_expr)).count()
        pct_val    = eliminated / fp_total * 100 if fp_total > 0 else 0

        if eliminated > best_eliminated:
            best_eliminated, best_label, best_tightened_pct = eliminated, label, pct

        cells.append(f"{eliminated} ({pct_val:.1f}%)".center(CW))

    print(f"  {label:<52}  " + "  ".join(cells))

print()

if best_label:

    best_pct_of_fp = best_eliminated / fp_total * 100 if fp_total > 0 else 0    
    print(f"      → eliminates {best_eliminated:,} / {fp_total:,} FP pairs  ({best_pct_of_fp:.1f}%)")
    print(f"  >>> Most impactful single change:")    
    print(f"      Tighten '{best_label}' by +{best_tightened_pct}%")

AND segments: 6
  name_closeness [OR]: ['name_score_jaccard >= 0.9', 'name_score_lcs >= 0.9', 'name_score_sbert >= 0.9', 'name_score_levenshtein >= 0.95', 'normalized_name_score_jaccard >= 0.95', 'normalized_name_score_lcs >= 0.95', 'normalized_name_score_levenshtein >= 0.95', 'normalized_name_score_sbert >= 0.95', 'average_name_score >= 0.8', 'average_normalized_name_score >= 0.9', 'normalized_name_score_containment >= 1', 'name_score_sbert >= 0.75', 'geo_distance_km <= 0.2', 'normalized_name_score_containment >= 1', 'name_score_sbert >= 0.7', 'geo_distance_km <= 0.1', 'name_score_containment >= 0.95', 'normalized_name_score_containment >= 0.95', 'star_ratings_score >= 0.8', 'geo_distance_km <= 0.2']
  address_distance_closeness [OR]: ['address_line1_score >= 0.5', 'address_sbert_score >= 0.5', 'geo_distance_km <= 0.2']
  property_type_closeness [LEAF]: ['property_type_score >= 0.5']
  sub_property_closeness [LEAF]: ['name_unit_score >= 0.5']
  house_number_closeness [OR]: ['address_u

FP pairs to analyse: 301

Sample (truncated to 40 chars):
+--------+--------+----------------------------------------+----------------------------------------+------------+------------+--------------+--------------+----------------------------------------+----------------------------------------+----------------------------------------+----------------------------------------+------------+------------+------------------+-----------------+-----------------+------+------+---------------------+-----------------+-------------------+-------------------------------+----------------------------+---------------------------+---------------------------+------------+-----------------------------+-------------------------+---------------------------+----------------------------------+-----------------------------------------+-------------------------------------+---------------------------------------------+---------------------------------------+-----------------------------+---------------------

In [10]:

# cell 10 — Export Run Report to Markdown
#
# Run this after cells 7, 8, 9 to snapshot the current config + results into a
# timestamped Markdown file.  Re-run with a different config.yaml and compare the
# two .md files side-by-side (VS Code diff, `diff`, or any text diffing tool).
#
# Output: ~/Downloads/hotel_mapping_reports/run_YYYYMMDD_HHMMSS.md

import yaml
import datetime
import os
from collections import Counter

CONFIG_PATH = "/Users/nakul.patil/Documents/hotel-mapping/src/hotel_data/config/config.yaml"
REPORTS_DIR = os.path.expanduser("~/Downloads/hotel_mapping_reports")
os.makedirs(REPORTS_DIR, exist_ok=True)

timestamp   = datetime.datetime.now().strftime("%Y%m%d_%H%M%S")
report_path = os.path.join(REPORTS_DIR, f"run_{timestamp}.md")

# ── Helpers ───────────────────────────────────────────────────────────────────
_CMP = {"gte": ">=", "lte": "<=", "gt": ">", "lt": "<", "eq": "=", "neq": "!="}

def _leaves(node):
    if "signal" in node:
        return [node]
    return [l for r in node["rules"] for l in _leaves(r)]

def _readable(node, indent=0):
    pad = "  " * indent
    if "signal" in node:
        return f"{pad}{node['signal']} {_CMP.get(node['comparator'], node['comparator'])} {node['threshold']}"
    op = node["operator"].upper()
    parts = [_readable(r, indent + 1) for r in node["rules"]]
    return f"{pad}(\n" + f"\n{pad}{op}\n".join(parts) + f"\n{pad})"

def _md_table(headers, rows, right_align_cols=None):
    """
    Render a Markdown table with every column padded to its max content width.
    right_align_cols: set of 0-based column indices to right-align (numbers, %).
    All rows are output as strings; caller should pre-format numbers.
    Returns a list of lines ready to extend into `lines`.
    """
    right_align_cols = right_align_cols or set()
    all_rows = [headers] + [[str(c) for c in r] for r in rows]
    widths = [max(len(str(all_rows[r][c])) for r in range(len(all_rows)))
              for c in range(len(headers))]

    def _fmt_row(row):
        parts = []
        for i, cell in enumerate(row):
            s = str(cell)
            parts.append(s.rjust(widths[i]) if i in right_align_cols else s.ljust(widths[i]))
        return "| " + " | ".join(parts) + " |"

    sep_parts = []
    for i, w in enumerate(widths):
        sep_parts.append(("-" * w + ":") if i in right_align_cols else ("-" * (w + 1)))
    separator = "|" + "|".join(sep_parts) + "|"

    out = [_fmt_row(headers), separator]
    for r in rows:
        out.append(_fmt_row([str(c) for c in r]))
    return out

with open(CONFIG_PATH) as _f:
    _cfg = yaml.safe_load(_f)
_match_logic = _cfg["scoring"]["match_logic"]
_group_nodes = _match_logic["rules"] if _match_logic.get("operator") == "AND" else [_match_logic]

lines = []

# ─────────────────────────────────────────────────────────────────────────────
# SECTION 1 — Header + config snapshot
# ─────────────────────────────────────────────────────────────────────────────
lines += [
    "# Hotel Cluster Analysis — Run Report",
    "",
    f"**Generated:** {datetime.datetime.now().strftime('%Y-%m-%d %H:%M:%S')}  ",
    f"**Config:** `{CONFIG_PATH}`",
    "",
    "---",
    "",
    "## WHERE Clause (current thresholds)",
    "",
    "```",
    _readable(_match_logic),
    "```",
    "",
]

# Threshold table  — Comparator | Threshold | Signal  (signal last = long col at end)
_seen = set()
_thr_rows = []
for _gnode in _group_nodes:
    for _leaf in _leaves(_gnode):
        _key = (_leaf["signal"], _leaf["comparator"], _leaf["threshold"])
        if _key not in _seen:
            _seen.add(_key)
            _c = _CMP.get(_leaf["comparator"], _leaf["comparator"])
            _thr_rows.append([_c, str(_leaf["threshold"]), f"`{_leaf['signal']}`"])

lines += _md_table(
    ["Comparator", "Threshold", "Signal"],
    _thr_rows,
    right_align_cols={0, 1},
)
lines += ["", "---", ""]

# ─────────────────────────────────────────────────────────────────────────────
# SECTION 2a — Alignment Check (rule_match vs cluster_id)
# ─────────────────────────────────────────────────────────────────────────────
lines += ["## Alignment Check  (rule_match vs cluster_id_i == cluster_id_j)", ""]
try:
    lines += _md_table(
        ["Metric", "Value"],
        [
            ["TP — rule=✓, cluster=same",  f"{aln_TP:,}"],
            ["FP — rule=✓, cluster=diff",  f"{aln_FP:,}"],
            ["FN — rule=✗, cluster=same",  f"{aln_FN:,}"],
            ["TN — rule=✗, cluster=diff",  f"{aln_TN:,}"],
            ["Total pairs",                f"{aln_total:,}"],
            ["Precision",                  f"{aln_precision:.4f}"],
            ["Recall",                     f"{aln_recall:.4f}"],
            ["F1",                         f"{aln_f1:.4f}"],
        ],
        right_align_cols={1},
    )
except NameError:
    lines.append("_Run cell 6 first._")
lines += ["", "---", ""]

# ─────────────────────────────────────────────────────────────────────────────
# SECTION 2b — Old vs New Cluster Logic (id_i==id_j vs cluster_id)
# ─────────────────────────────────────────────────────────────────────────────
lines += ["## Old vs New Cluster Logic  (id_i==id_j vs cluster_id_i==cluster_id_j)", ""]
try:
    lines += _md_table(
        ["Metric", "Value"],
        [
            ["TP — both agree: match",      f"{TP:,}"],
            ["FN — old matched, new missed", f"{FN:,}"],
            ["FP — old missed, new matched", f"{FP:,}"],
            ["TN — both agree: no match",    f"{TN:,}"],
            ["Total pairs",                  f"{total:,}"],
            ["Precision",                    f"{precision:.4f}"],
            ["Recall",                       f"{recall:.4f}"],
            ["F1",                           f"{f1:.4f}"],
        ],
        right_align_cols={1},
    )
except NameError:
    lines.append("_Run cell 9 first._")
lines += ["", "---", ""]

# ─────────────────────────────────────────────────────────────────────────────
# SECTION 3 — FN analysis
# ─────────────────────────────────────────────────────────────────────────────
lines += ["## FN Condition Analysis", ""]
try:
    _fn_pass_cols   = [c for c in fn_pdf.columns if c.endswith("_pass") and c != "where_clause_pass"]
    _fn_margin_cols = [c for c in fn_pdf.columns if "_margin_" in c]
    _fn_total       = len(fn_pdf)

    lines.append(f"**Total FN pairs:** {_fn_total:,}  ")
    lines.append("")
    lines.append("### Failure Attribution")
    lines.append("")

    _combo_ctr = Counter()
    for _, _row in fn_pdf.iterrows():
        _failing = [c.replace("_pass", "") for c in _fn_pass_cols if _row.get(c) == False]
        _combo_ctr[" + ".join(_failing) if _failing else "none"] += 1

    _attr_rows = []
    for _combo, _cnt in sorted(_combo_ctr.items(), key=lambda x: -x[1]):
        _pct = _cnt / _fn_total * 100
        _n   = len(_combo.split(" + ")) if _combo != "none" else 0
        _note = "possible bad ground-truth" if _n >= 3 else ""
        _attr_rows.append([_combo, f"{_cnt:,}", f"{_pct:.1f}%", str(_n), _note])

    lines += _md_table(
        ["Failing Group Combination", "Count", "%", "#Groups", "Note"],
        _attr_rows,
        right_align_cols={1, 2, 3},
    )

    _bad = sum(v for k, v in _combo_ctr.items() if k != "none" and len(k.split(" + ")) >= 3)
    lines += [
        "",
        f"**Rows with ≥3 groups failing** (likely bad ground-truth): {_bad:,}  ({_bad/_fn_total*100:.1f}%)  ",
        f"**Rows with 1–2 groups failing** (tunable): {_fn_total-_bad:,}  ({(_fn_total-_bad)/_fn_total*100:.1f}%)",
        "",
        "### Closest Signal Frequency",
        "",
    ]

    _freq_rows = [
        [f"`{_mc}`", f"{fn_pdf[_mc].notna().sum():,}", f"{fn_pdf[_mc].notna().sum()/_fn_total*100:.1f}%"]
        for _mc in _fn_margin_cols
    ]
    lines += _md_table(
        ["Margin Column", "FN rows closest to threshold", "%"],
        _freq_rows,
        right_align_cols={1, 2},
    )
    lines += ["", "---", ""]

except NameError:
    lines += ["_Run cell 8 first._", "", "---", ""]

# ─────────────────────────────────────────────────────────────────────────────
# SECTION 4 — FP analysis
# ─────────────────────────────────────────────────────────────────────────────
lines += ["## FP Condition Analysis", ""]
try:
    _fp_margin_cols = [c for c in fp_pdf.columns if "_margin_" in c]
    _fp_total       = len(fp_pdf)

    lines.append(f"**Total FP pairs:** {_fp_total:,}  ")
    lines.append("")
    lines.append("### Most Fragile Signal per FP Pair")
    lines.append("")

    try:
        _frag_rows = [
            [
                f"`{_row['most_fragile_col']}`",
                str(_row['most_fragile_group']),
                f"{_row['single_pct']}%",
                f"{_row['count']:,}",
                f"{_row['pct']:.1f}%",
                f"{_row['avg_margin']:.4f}",
            ]
            for _, _row in table_data.iterrows()
        ]
    except NameError:
        _frag_rows = [["_Run cell 9 to populate_", "", "", "", "", ""]]

    lines += _md_table(
        ["Signal (most fragile)", "Group", "1-leaf%", "Count", "%", "Avg margin"],
        _frag_rows,
        right_align_cols={2, 3, 4, 5},
    )

    try:
        _single = int(fp_pdf2["is_single_or_pass"].sum())
        lines += [
            "",
            f"**Single-signal fix sufficient:** {_single:,}  ({_single/_fp_total*100:.1f}%)  ",
            f"**Multi-signal fix needed:** {_fp_total-_single:,}  ({(_fp_total-_single)/_fp_total*100:.1f}%)",
        ]
    except NameError:
        pass
    lines.append("")

    lines.append("### Closest Signal Frequency")
    lines.append("")

    _fp_freq_rows = [
        [f"`{_mc}`", f"{fp_pdf[_mc].notna().sum():,}", f"{fp_pdf[_mc].notna().sum()/_fp_total*100:.1f}%"]
        for _mc in _fp_margin_cols
    ]
    lines += _md_table(
        ["Margin Column", "FP rows barely above threshold", "%"],
        _fp_freq_rows,
        right_align_cols={1, 2},
    )
    lines += ["", "---", ""]

except NameError:
    lines += ["_Run cell 9 first._", "", "---", ""]

# ─────────────────────────────────────────────────────────────────────────────
# Write & preview
# ─────────────────────────────────────────────────────────────────────────────
_report_md = "\n".join(lines)
with open(report_path, "w") as _f:
    _f.write(_report_md)

print(f"✓ Report saved → {report_path}")
print()
print("To compare two runs:")
print(f"  code --diff ~/Downloads/hotel_mapping_reports/<run_A>.md \\")
print(f"              ~/Downloads/hotel_mapping_reports/<run_B>.md")
print()
print("--- Preview (first 60 lines) ---")
print("\n".join(lines[:60]))

✓ Report saved → /Users/nakul.patil/Downloads/hotel_mapping_reports/run_20260314_212352.md

To compare two runs:
  code --diff ~/Downloads/hotel_mapping_reports/<run_A>.md \
              ~/Downloads/hotel_mapping_reports/<run_B>.md

--- Preview (first 60 lines) ---
# Hotel Cluster Analysis — Run Report

**Generated:** 2026-03-14 21:23:52  
**Config:** `/Users/nakul.patil/Documents/hotel-mapping/src/hotel_data/config/config.yaml`

---

## WHERE Clause (current thresholds)

```
(
  (
    name_score_jaccard >= 0.9
  OR
    name_score_lcs >= 0.9
  OR
    name_score_sbert >= 0.9
  OR
    name_score_levenshtein >= 0.95
  OR
    normalized_name_score_jaccard >= 0.95
  OR
    normalized_name_score_lcs >= 0.95
  OR
    normalized_name_score_levenshtein >= 0.95
  OR
    normalized_name_score_sbert >= 0.95
  OR
    average_name_score >= 0.8
  OR
    average_normalized_name_score >= 0.9
  OR
    (
      normalized_name_score_containment >= 1
    AND
      name_score_sbert >= 0.75
    AND
      ge